# eval_funnel_recall — per-stage recall waterfall

Step 1 of the reflective-labeler build (`docs/reflective_labeler_design.md` §9.1). **No API spend.**

**Question:** as the funnel narrows the ~1000-doc hybrid pool toward the ~W docs the Claude labeler
will actually see, how much of the *relevant set survives each stage*, and what NDCG@10 ceiling does
that leave the labeler? The funnel's only job is to not drop relevant trials before the expensive
labeler sees them (design §5.0, §6).

Stages: **hybrid retrieval pool → clf-R cross-encoder rerank → top-W.** Reuses the cached pool from
`eval_fullcorpus.ipynb` when present; otherwise rebuilds it (BM25 + dense + RRF).

## Setup (Colab — GPU for the clf rerank)

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q rank-bm25 pytrec_eval sentence-transformers datasets pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, pickle
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from ctmatch.experiments import (ExperimentConfig, load_corpus, load_eval, build_bm25, encode_corpus,
                                 rrf_fuse, cross_encoder_scores, relevant_index, ndcg_at_k, recall_at_k)
cfg = ExperimentConfig(data_root=DATA_ROOT)
CAND_K = cfg.cand_k   # 1000
SETS = ['trec21', 'kz', 'trec22']
os.makedirs(cfg.path('cache'), exist_ok=True); os.makedirs(cfg.path('data'), exist_ok=True)
print('repr:', cfg.repr_tag(), '| clf:', cfg.clf_ckpt)

In [ ]:
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, SETS)
topics = {s: [t for t in sets[s]['rel_dict'] if t in sets[s]['topic2text']] for s in SETS}
print(f'{len(corpus_ids):,} docs |', {s: len(v) for s, v in topics.items()}, 'topics')

## Stage 0 — hybrid retrieval pool

Load the pool `eval_fullcorpus.ipynb` already wrote; rebuild only if absent. The rebuild reuses the
cached BM25 index and dense embeddings, so the expensive parts run at most once.

In [ ]:
pool_path = cfg.path('data/pool_R.json')
if os.path.exists(pool_path):
    pool = json.load(open(pool_path))
    print('loaded cached pool:', pool_path)
else:
    print('no cached pool — rebuilding (BM25 + dense + RRF)')
    from sentence_transformers import SentenceTransformer
    bm25_path = cfg.bm25_file('fulltext_R')
    if os.path.exists(bm25_path):
        bm25 = pickle.load(open(bm25_path, 'rb'))
    else:
        bm25 = build_bm25(corpus_fields, cfg); pickle.dump(bm25, open(bm25_path, 'wb'))
    emb_path = cfg.emb_file('fulltext_R')
    if os.path.exists(emb_path):
        doc_emb = np.load(emb_path)
    else:
        doc_emb = encode_corpus(corpus_fields, cfg); np.save(emb_path, doc_emb)
    q_enc = SentenceTransformer(cfg.retriever_ckpt); q_enc.max_seq_length = cfg.retriever_max_tokens
    def bm25_topk(qt, k):
        sc = bm25.get_scores(qt.lower().split()); top = np.argpartition(-sc, k)[:k]; top = top[np.argsort(-sc[top])]
        return {corpus_ids[i]: float(sc[i]) for i in top}
    def dense_topk(qt, k):
        q = q_enc.encode([qt], normalize_embeddings=True)[0].astype('float32'); sims = doc_emb @ q
        top = np.argpartition(-sims, k)[:k]; top = top[np.argsort(-sims[top])]
        return {corpus_ids[i]: float(sims[i]) for i in top}
    pool = {}
    for s in SETS:
        pool[s] = {}
        for t in tqdm(topics[s], desc=f'retrieve {s}'):
            qt = sets[s]['topic2text'][t]; bm, dn = bm25_topk(qt, CAND_K), dense_topk(qt, CAND_K)
            rrf = rrf_fuse([sorted(bm, key=bm.get, reverse=True), sorted(dn, key=dn.get, reverse=True)], k=cfg.rrf_k)
            pool[s][t] = sorted(set(bm) | set(dn), key=lambda d: rrf.get(d, 0), reverse=True)
    json.dump(pool, open(pool_path, 'w')); print('wrote', pool_path)

## Stage 1 — clf-R cross-encoder rerank of each pool (cached)

Scores every pooled doc with `ctmatch-clf-R` (the frozen eligibility-view cross-encoder) under the
same `repr_strategy`/`max_length` it was trained on. `clf_order` is the ranking the labeler's top-W
input would be drawn from.

In [ ]:
# Lazy clf loader — only invoked on a cache miss, so reusing ce_clf_R.npz skips the model download.
_clf = {}
def load_clf():
    if _clf:
        return _clf['model'], _clf['tok'], _clf['rel_idx']
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    m = AutoModelForSequenceClassification.from_pretrained(cfg.clf_ckpt).to(dev).eval()
    tok = AutoTokenizer.from_pretrained(cfg.clf_ckpt)
    ri = relevant_index(m)
    print('loaded clf:', cfg.clf_ckpt, '| id2label:', m.config.id2label, '| rel_idx:', ri)
    _clf.update(model=m, tok=tok, rel_idx=ri)
    return m, tok, ri

In [ ]:
# clf_scores[s][t][d] = P(relevant). Prefer the ensemble's cached CE scores over pool_R
# (cache/ce_clf_R.npz, keyed (source,topic,doc)->(P_rel,P_partial)); recompute only if the
# cache is absent OR its keys don't cover pool_R (guards the §2h ce_* naming-drift hazard).
clf_scores = None
ce_path = cfg.ce_cache_path('clf')   # -> cache/ce_clf_R.npz for pool_tag='R'
if os.path.exists(ce_path):
    ce_d = np.load(ce_path, allow_pickle=True)['d'].item()   # {(s,t,d): (P_rel, P_partial)}
    cov = {s: float(np.mean([(s, t, pool[s][t][0]) in ce_d for t in topics[s]])) for s in SETS}
    print('cached CE coverage (top-doc hit rate) by split:', {s: round(c, 3) for s, c in cov.items()})
    if all(c > 0.99 for c in cov.values()):
        tset = {s: set(topics[s]) for s in SETS}
        clf_scores = {s: {} for s in SETS}
        for (s, t, d), (rr, _pp) in ce_d.items():
            if s in clf_scores and t in tset[s]:
                clf_scores[s].setdefault(t, {})[d] = rr
        print('REUSING cached clf-R CE scores:', ce_path)
    else:
        print('cache keys do not align to pool_R -> recomputing (drift guard tripped)')

if clf_scores is None:
    clf_path = cfg.path('cache/clf_pool_scores_R.pkl')
    if os.path.exists(clf_path):
        clf_scores = pickle.load(open(clf_path, 'rb')); print('loaded local recompute cache:', clf_path)
    else:
        clf, clf_tok, rel_idx = load_clf()
        clf_scores = {}
        for s in SETS:
            clf_scores[s] = {}
            for t in tqdm(topics[s], desc=f'clf {s}'):
                docs = pool[s][t]; fields = [id2fields[d] for d in docs]
                sc = cross_encoder_scores(clf, clf_tok, sets[s]['topic2text'][t], fields, cfg, rel_idx)
                clf_scores[s][t] = dict(zip(docs, sc))
        pickle.dump(clf_scores, open(clf_path, 'wb')); print('wrote', clf_path)

# Rank each pool by P(relevant). Any pool doc missing a score (cache gaps) sorts last.
clf_order = {s: {t: sorted(pool[s][t], key=lambda d: clf_scores[s].get(t, {}).get(d, -1.0), reverse=True)
                 for t in topics[s]} for s in SETS}

## The recall waterfall

`ret@W` = recall in the RRF pool order (retrieval alone). `clf@W` = recall after the clf rerank — the
order the labeler's top-W is drawn from. **The load-bearing comparison at small W:** if `clf@W` <
`ret@W`, the clf stage is *burying* relevant trials as it narrows (the §7i extraction problem) and we
should widen W or fuse clf with the hybrid order (§5.4); if `clf@W` ≥ `ret@W`, clf pulls relevant up
and narrowing is safe. `rel_level=2` = Eligible-only; `1` = Eligible+Excluded.

In [ ]:
WIDTHS = [25, 50, 100, 500, 1000]
def mean(xs): return float(np.mean(xs))

rec_rows = []
for s in SETS:
    rel = sets[s]['rel_dict']
    for lvl, name in [(2, 'elig'), (1, 'elig+exc')]:
        row = {'split': s, 'rel': name}
        for W in WIDTHS:
            row[f'ret@{W}'] = round(mean([recall_at_k(pool[s][t], rel[t], W, rel_level=lvl) for t in topics[s]]), 3)
            row[f'clf@{W}'] = round(mean([recall_at_k(clf_order[s][t], rel[t], W, rel_level=lvl) for t in topics[s]]), 3)
        rec_rows.append(row)
recall_df = pd.DataFrame(rec_rows)
recall_df

## Labeler ceiling by width

`clf_topW_oracle` = NDCG@10 if the labeler perfectly scored clf's top-W (i.e. the best the funnel
*leaves achievable* at that width). Compare to `pool_oracle` (≈ the ~0.957 whole-pool ceiling from
`eval_fullcorpus`). The knee — the smallest W where the ceiling stops dropping — is the cheapest
labeler width that doesn't starve accuracy (design §10 open question: funnel width W).

In [ ]:
LAB_W = [25, 50, 100, 500]
def oracle_topW(order, rel, W):
    return ndcg_at_k(sorted(order[:W], key=lambda d: rel.get(d, 0), reverse=True), rel)

cei_rows = []
for s in SETS:
    rel = sets[s]['rel_dict']
    row = {'split': s,
           'pool_oracle': round(mean([oracle_topW(pool[s][t], rel[t], len(pool[s][t])) for t in topics[s]]), 3)}
    for W in LAB_W:
        row[f'clf_top{W}_oracle'] = round(mean([oracle_topW(clf_order[s][t], rel[t], W) for t in topics[s]]), 3)
    cei_rows.append(row)
ceiling_df = pd.DataFrame(cei_rows)
ceiling_df

In [ ]:
# Persist for the next step (label_pointwise_opus.ipynb) and for the paper.
recall_df.to_csv(cfg.path('data/funnel_recall_R.csv'), index=False)
ceiling_df.to_csv(cfg.path('data/funnel_ceiling_R.csv'), index=False)
json.dump({s: clf_order[s] for s in SETS}, open(cfg.path('data/clf_order_R.json'), 'w'))
print('wrote data/funnel_recall_R.csv, data/funnel_ceiling_R.csv, data/clf_order_R.json')

## How to read this (feeds the design decision)

- **TREC22 is the one that matters** for the app claim; TREC21 corroborates; KZ is expected to lag
  (2015-qrels-vs-2021-corpus + terse topics — design §4).
- Pick the labeler width **W** at the knee of `clf_top{W}_oracle`: the smallest W whose ceiling is
  still near `pool_oracle`. That W sets the labeler's calls/topic (cost & latency) in §8.
- If `clf@{25,50}` is well below `ret@{25,50}`, do **not** narrow with clf alone — feed the labeler
  a clf+hybrid fused / union top-W so recall isn't thrown away before it (§5.4).